In [1]:
import pandas as pd                                                                                                                                                                                
import glob                                                                                                                                                                                        
import os                                                                                                                                                                                          
                                                                                                                                                                                                    
path = '../data'                                                                                                                                                                                   
all_files = glob.glob(path + '/parquet/**/*.parquet', recursive=True)                                                                                                                              
                                                                                                                                                                                                    
dfs = []                                                                                                                                                                                           
for f in all_files:                                                                                                                                                                                
    temp = pd.read_parquet(f)                                                                                                                                                                       
    dfs.append(temp)                                                                                                                                                                               
                                                                                                                                                                                                    
df = pd.concat(dfs, ignore_index=True)                                                                                                                                                             
                                                                                                                                                                                                    
print("Shape:", df.shape)                                                                                                                                                                          
print("\nColumns:", df.columns.tolist())                                                                                                                                                           
print("\nLabel distribution:")                                                                                                                                                                     
print(df['Label'].value_counts())   

Shape: (2313810, 77)

Columns: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag 

In [2]:
df['target'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print(df['target'].value_counts())
print(f"\nImbalance ratio: {df['target'].value_counts()[0] / df['target'].value_counts()[1]:.2f}:1")

target
0    1977318
1     336492
Name: count, dtype: int64

Imbalance ratio: 5.88:1


In [3]:
import numpy as np

print("Null values:", df.isnull().sum().sum())
print("Infinite values:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())

print("\nColumns with nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nColumns with infinite values:")
numeric_cols = df.select_dtypes(include=np.number).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
print(inf_counts[inf_counts > 0])

Null values: 0
Infinite values: 0

Columns with nulls:
Series([], dtype: int64)

Columns with infinite values:
Series([], dtype: int64)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features and target
drop_cols = ['Label', 'target']
X = df.drop(columns=drop_cols)
y = df['target']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain label distribution:")
print(y_train.value_counts())
print("\nTest label distribution:")
print(y_test.value_counts())

Train size: (1851048, 76)
Test size: (462762, 76)

Train label distribution:
target
0    1581854
1     269194
Name: count, dtype: int64

Test label distribution:
target
0    395464
1     67298
Name: count, dtype: int64


In [7]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("After SMOTE:")
print("Train size:", X_train_resampled.shape)
import pandas as pd
print(pd.Series(y_train_resampled).value_counts())

After SMOTE:
Train size: (3163708, 76)
target
0    1581854
1    1581854
Name: count, dtype: int64


In [8]:
from sklearn.ensemble import RandomForestClassifier
import time

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1  # uses all available CPU cores
)

start = time.time()
rf.fit(X_train_resampled, y_train_resampled)
end = time.time()

print(f"Training time: {(end - start):.2f} seconds")

Training time: 539.12 seconds


In [9]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

# Predict on test set
y_pred = rf.predict(X_test_scaled)
y_pred_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

TN, FP, FN, TP = cm.ravel()
FPR = FP / (FP + TN)
FNR = FN / (FN + TP)

print(f"\nFalse Positive Rate (FPR): {FPR:.4f}")
print(f"False Negative Rate (FNR): {FNR:.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00    395464
      Attack       1.00      1.00      1.00     67298

    accuracy                           1.00    462762
   macro avg       1.00      1.00      1.00    462762
weighted avg       1.00      1.00      1.00    462762


Confusion Matrix:
[[395128    336]
 [   263  67035]]

False Positive Rate (FPR): 0.0008
False Negative Rate (FNR): 0.0039
AUC-ROC: 1.0000


In [10]:
import pandas as pd

feature_names = X.columns.tolist()
importances = rf.feature_importances_

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(feat_imp.head(20))

                     feature  importance
41    Packet Length Variance    0.085238
12     Bwd Packet Length Std    0.083597
40         Packet Length Std    0.074733
11    Bwd Packet Length Mean    0.065294
51           Avg Packet Size    0.064313
9      Bwd Packet Length Max    0.057199
53      Avg Bwd Segment Size    0.053428
2     Total Backward Packets    0.032728
38         Packet Length Max    0.031932
1          Total Fwd Packets    0.031904
16              Flow IAT Std    0.026886
4   Bwd Packets Length Total    0.025033
63         Subflow Bwd Bytes    0.024376
39        Packet Length Mean    0.023535
60       Subflow Fwd Packets    0.023508
65        Init Bwd Win Bytes    0.016891
62       Subflow Bwd Packets    0.016489
64        Init Fwd Win Bytes    0.014688
52      Avg Fwd Segment Size    0.014595
34         Bwd Header Length    0.014517


In [11]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import numpy as np
import pandas as pd

# Evaluate
y_pred = rf.predict(X_test_scaled)
y_pred_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Classification report
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
FPR = FP / (FP + TN)
FNR = FN / (FN + TP)
print(f"FPR: {FPR:.4f}")
print(f"FNR: {FNR:.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Feature importance
feature_names = X.columns.tolist()
feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 20 features:")
print(feat_imp.head(20))

              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00    395464
      Attack       1.00      1.00      1.00     67298

    accuracy                           1.00    462762
   macro avg       1.00      1.00      1.00    462762
weighted avg       1.00      1.00      1.00    462762

FPR: 0.0008
FNR: 0.0039
AUC-ROC: 1.0000

Top 20 features:
                     feature  importance
41    Packet Length Variance    0.085238
12     Bwd Packet Length Std    0.083597
40         Packet Length Std    0.074733
11    Bwd Packet Length Mean    0.065294
51           Avg Packet Size    0.064313
9      Bwd Packet Length Max    0.057199
53      Avg Bwd Segment Size    0.053428
2     Total Backward Packets    0.032728
38         Packet Length Max    0.031932
1          Total Fwd Packets    0.031904
16              Flow IAT Std    0.026886
4   Bwd Packets Length Total    0.025033
63         Subflow Bwd Bytes    0.024376
39        Packet Length Mean    0.023535


In [16]:
# Save
from sklearn.pipeline import Pipeline
import joblib

rf_pipeline = Pipeline([
    ("scaler", scaler),
    ("rf", rf)
])

joblib.dump(rf_pipeline, "./rf_pipeline.pkl")
np.save('./X_test_scaled.npy', X_test_scaled)
np.save('./y_test.npy', np.array(y_test))
feat_imp.to_csv('./feature_importance.csv', index=False)
print("\nAll saved.")


All saved.


In [17]:
from sklearn.model_selection import cross_val_score

X_cv_sample = X_train_resampled[:50000]
y_cv_sample = y_train_resampled[:50000]

cv_scores = cross_val_score(rf, X_cv_sample, y_cv_sample,
                             cv=5, scoring='f1', n_jobs=-1)
print("CV F1 scores:", cv_scores)
print("Mean:", cv_scores.mean())
print("Std:", cv_scores.std())

CV F1 scores: [0.99182004 0.98872566 0.99043062 0.98841172 0.98844324]
Mean: 0.9895662562162654
Std: 0.0013511030618795338


In [18]:
for i, col in enumerate(X.columns.tolist()):
    print(i, col)

0 Flow Duration
1 Total Fwd Packets
2 Total Backward Packets
3 Fwd Packets Length Total
4 Bwd Packets Length Total
5 Fwd Packet Length Max
6 Fwd Packet Length Min
7 Fwd Packet Length Mean
8 Fwd Packet Length Std
9 Bwd Packet Length Max
10 Bwd Packet Length Min
11 Bwd Packet Length Mean
12 Bwd Packet Length Std
13 Flow Bytes/s
14 Flow Packets/s
15 Flow IAT Mean
16 Flow IAT Std
17 Flow IAT Max
18 Flow IAT Min
19 Fwd IAT Total
20 Fwd IAT Mean
21 Fwd IAT Std
22 Fwd IAT Max
23 Fwd IAT Min
24 Bwd IAT Total
25 Bwd IAT Mean
26 Bwd IAT Std
27 Bwd IAT Max
28 Bwd IAT Min
29 Fwd PSH Flags
30 Bwd PSH Flags
31 Fwd URG Flags
32 Bwd URG Flags
33 Fwd Header Length
34 Bwd Header Length
35 Fwd Packets/s
36 Bwd Packets/s
37 Packet Length Min
38 Packet Length Max
39 Packet Length Mean
40 Packet Length Std
41 Packet Length Variance
42 FIN Flag Count
43 SYN Flag Count
44 RST Flag Count
45 PSH Flag Count
46 ACK Flag Count
47 URG Flag Count
48 CWE Flag Count
49 ECE Flag Count
50 Down/Up Ratio
51 Avg Packet Siz